In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split

SEED = 42
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

SIGNALS = ["signal1", "signal2", "signal3"]
COLORS = ["#0072B2", "#E69F00", "#009E73"]  # Okabe-Ito, colorblind-safe

print(f"scikit-learn {sklearn.__version__}")  # the split is reproducible for a given version

scikit-learn 1.9.1


In [9]:
df = pd.read_csv(RAW_DIR/"Round1_Diagnostic_33.csv")

# Assuming your dataframe is named `df`
# Columns present: id, own, competing1, competing2, borrowertype, accepted, default, profit

# 1. Determine acceptance / winner based on borrowertype preference
# Removing 2% (2 percentage points) from the rate of the lender matching the borrowertype
adj_own = df['own'] - np.where(df['borrowertype'] == 3, 0.02, 0.0)
adj_comp1 = df['competing1'] - np.where(df['borrowertype'] == 1, 0.02, 0.0)
adj_comp2 = df['competing2'] - np.where(df['borrowertype'] == 2, 0.02, 0.0)

adj_rates = pd.DataFrame({
    'own': adj_own,
    'competing1': adj_comp1,
    'competing2': adj_comp2
})

# The borrower chooses the lender with the lowest adjusted rate
winner = adj_rates.idxmin(axis=1)

df['accepted'] = (winner == 'own').astype(int)
df['accepted1'] = (winner == 'competing1').astype(int)
df['accepted2'] = (winner == 'competing2').astype(int)

# 2. Get the actual interest rate charged by the winning lender
winning_rate = np.select(
    [winner == 'own', winner == 'competing1', winner == 'competing2'],
    [df['own'], df['competing1'], df['competing2']],
    default=0.0
)

# 3. Calculate profits based on default status and winning rate
# If default == 1: -10,000
# If default == 0: 10,000 * winning_rate
base_profit = np.where(
    df['default'] == 1,
    -10000.0,
    10000.0 * winning_rate
)

# Assign profits only to the winning lender (0 for losers)
df['profit'] = np.where(df['accepted'] == 1, base_profit, 0.0)
df['profit1'] = np.where(df['accepted1'] == 1, base_profit, 0.0)
df['profit2'] = np.where(df['accepted2'] == 1, base_profit, 0.0)

# 4. Total P&L summary
print(f"Our Total P&L: {df['profit'].sum():,.2f} €")
print(f"Competitor 1 Total P&L: {df['profit1'].sum():,.2f} €")
print(f"Competitor 2 Total P&L: {df['profit2'].sum():,.2f} €")

Our Total P&L: 30,908,296.05 €
Competitor 1 Total P&L: 7,771,402.51 €
Competitor 2 Total P&L: 7,491,141.47 €


C:\Users\gieth\AppData\Local\Temp\ipykernel_47020\463520593.py:19: FutureWarning: The behavior of DataFrame.idxmin with all-NA values, or any-NA and skipna=False, is deprecated. In a future version this will raise ValueError
  winner = adj_rates.idxmin(axis=1)


In [16]:
df.head(50)

,id,own,competing1,competing2,borrowertype,accepted,default,profit,accepted1,accepted2,profit1,profit2
0,1,0.057185,0.039284,0.055900,3,1,0,571.85002,0,0,0.00000,0.00000
1,2,0.223785,NaN,NaN,1,1,1,-10000.00000,0,0,0.00000,0.00000
2,3,0.054445,0.043756,0.073900,3,1,0,544.44999,0,0,0.00000,0.00000
3,4,0.153655,0.115800,NaN,3,0,0,0.00000,1,0,1158.00000,0.00000
4,5,0.297273,0.159800,NaN,1,0,0,0.00000,1,0,1597.99990,0.00000
5,6,0.256758,0.245529,0.193300,3,0,0,0.00000,0,1,0.00000,1932.99990
6,7,0.029253,0.044000,0.031900,1,0,0,0.00000,1,0,440.00000,0.00000
7,8,0.502280,0.919900,0.580800,3,1,0,5022.80000,0,0,0.00000,0.00000
8,9,0.044969,NaN,0.038900,2,0,0,0.00000,0,1,0.00000,388.99999
9,10,0.270106,NaN,0.178900,2,0,0,0.00000,0,1,0.00000,1789.00000


In [11]:
adj_rates.head(10)

,own,competing1,competing2
0,0.037185,0.039284,0.0559
1,0.223785,NaN,NaN
2,0.034445,0.043756,0.0739
3,0.133655,0.115800,NaN
4,0.297273,0.139800,NaN
5,0.236758,0.245529,0.1933
6,0.029253,0.024000,0.0319
7,0.482280,0.919900,0.5808
8,0.044969,NaN,0.0189
9,0.270106,NaN,0.1589


In [15]:
import pandas as pd

# 1. Look at the distribution of your interest rates ('own')
print("Interest rate summary:")
print(df['own'].describe())
print("-" * 40)

# 2. Filter for applicants who received the highest interest rates (e.g., top 10%)
threshold = df['own'].quantile(0.90)
high_rate_applicants = df[df['own'] >= threshold]

# 3. Analyze their default rate
high_rate_defaults = high_rate_applicants['default'].mean()
overall_defaults = df['default'].mean()

print(f"Threshold for highest rates (Top 10%): {threshold:.2f}%")
print(f"Number of applicants in this high-rate bracket: {len(high_rate_applicants):,}")
print(f"Default rate for high-rate applicants: {high_rate_defaults * 100:.2f}%")
print(f"Overall portfolio default rate: {overall_defaults * 100:.2f}%")

Interest rate summary:
count    99856.000000
mean         0.202590
std          0.161338
min          0.010215
25%          0.071114
50%          0.169530
75%          0.290340
max          0.995898
Name: own, dtype: float64
----------------------------------------
Threshold for highest rates (Top 10%): 0.42%
Number of applicants in this high-rate bracket: 9,986
Default rate for high-rate applicants: 24.77%
Overall portfolio default rate: 9.95%


In [17]:
import pandas as pd

# Assuming your dataframe is named `df` and contains 'accepted' and 'default' columns

# 1. Filter for applicants you won (where accepted == 1)
won_applicants = df[df['accepted'] == 1]

# 2. Compute the default rate among the won applicants
if len(won_applicants) > 0:
    default_portion = won_applicants['default'].mean()
    print(f"Total applicants won: {len(won_applicants):,}")
    print(f"Total defaulter applicants won: {won_applicants['default'].sum():,}")
    print(f"Portion of won applicants who defaulted: {default_portion * 100:.2f}%")
else:
    print("No applicants were won.")

Total applicants won: 49,159
Total defaulter applicants won: 6,020
Portion of won applicants who defaulted: 12.25%
